In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Lodhi Road, Delhi - IITM.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,...,Toluene,Eth-Benzene,MP-Xylene,RH,WS,WD,BP,Xylene,AT,RF
0,01-01-2025 00:00,02-01-2025 00:00,148.86,296.19,NaN,64.10,NaN,NaN,20.81,NaN,...,NaN,NaN,NaN,89.25,0.73,136.39,NaN,NaN,NaN,0.0
1,02-01-2025 00:00,03-01-2025 00:00,122.12,245.69,NaN,63.41,NaN,NaN,17.87,NaN,...,NaN,NaN,NaN,92.14,0.76,139.32,NaN,NaN,NaN,0.0
2,03-01-2025 00:00,04-01-2025 00:00,190.83,375.22,NaN,71.72,NaN,NaN,23.24,NaN,...,NaN,NaN,NaN,86.11,0.86,151.45,NaN,NaN,NaN,0.0
3,04-01-2025 00:00,05-01-2025 00:00,195.5,384.31,NaN,69.24,NaN,NaN,23.43,NaN,...,NaN,NaN,NaN,92.45,0.71,146.15,NaN,NaN,NaN,0.0
4,05-01-2025 00:00,06-01-2025 00:00,145.05,290.61,NaN,60.10,NaN,NaN,18.18,NaN,...,NaN,NaN,NaN,96.57,0.47,154.10,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,12-11-2025 00:00,13-11-2025 00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
641,13-11-2025 00:00,14-11-2025 00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
642,14-11-2025 00:00,15-11-2025 00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
643,15-11-2025 00:00,16-11-2025 00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (645, 12)


In [6]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NO', 'Benzene']
Dropped rows (>70% NaN): 15
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO2          0
SO2          0
RH           0
WS           0
WD           0
RF           0
dtype: int64


In [7]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [8]:




# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (630, 10)
          From Date           To Date   PM2.5    PM10     NO2    SO2     RH  \
0  01-01-2025 00:00  02-01-2025 00:00  148.86  116.91  65.155  13.87  72.37   
1  02-01-2025 00:00  03-01-2025 00:00  122.12  116.91  65.155  13.87  72.37   
2  03-01-2025 00:00  04-01-2025 00:00  190.83  116.91  65.155  13.87  72.37   
3  04-01-2025 00:00  05-01-2025 00:00   195.5  116.91  65.155  13.87  72.37   
4  05-01-2025 00:00  06-01-2025 00:00  145.05  116.91  65.155  13.87  72.37   

     WS      WD   RF  
0  1.04  245.99  0.0  
1  1.04  245.99  0.0  
2  1.04  245.99  0.0  
3  1.04  245.99  0.0  
4  1.04  245.99  0.0  


In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [10]:
df

,From Date,To Date,PM2.5,PM10,NO2,SO2,RH,WS,WD,RF
0,01-01-2025 00:00,02-01-2025 00:00,148.86,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,122.12,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,190.83,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,195.5,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,145.05,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
625,12-11-2025 00:00,13-11-2025 00:00,0,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0
626,13-11-2025 00:00,14-11-2025 00:00,0,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0
627,14-11-2025 00:00,15-11-2025 00:00,0,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0
628,15-11-2025 00:00,16-11-2025 00:00,0,1.421085e-14,2.842171e-14,0.0,0.0,0.0,0.0,0.0


In [11]:
df.to_excel('lodhiroadIITM2025.xlsx', index=False)